# V14.5 - Gate Sweep Full Train Valid Select

Train answer-only LoRA on all cleaned train rows and select the epoch directly on valid.json, with per-type retrieval gate sweep enabled for the V14 gate-sweep policy.


In [1]:
# ============================================================
# 0. Install/import dependencies
# ============================================================
import os, sys, json, math, time, re, random, hashlib, inspect, shutil, unicodedata
from pathlib import Path
from collections import Counter, defaultdict, deque
from dataclasses import dataclass

PIP_INSTALL_DEPS = False  # Kaggle official run: Internet OFF; avoid pip overhead
PIP_PACKAGES = ["peft", "accelerate", "datasets", "evaluate"]

if PIP_INSTALL_DEPS:
    import subprocess
    print("[pip] Installing:", " ".join(PIP_PACKAGES))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES])

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
from tqdm.auto import tqdm

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainerCallback,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
import peft

print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))
print("PEFT :", getattr(peft, "__version__", "unknown"))


Torch: 2.10.0+cu128
CUDA : True | GPU count: 2
0 Tesla T4
1 Tesla T4
PEFT : 0.18.1


In [2]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Cannot find any path: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/dataset-math",
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "dataset",
)

MODEL_NAME = str(first_existing(
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "GPT2_vietnamese",
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"

# v14_5_gate_sweep_full_train_valid_select: answer-only optimization for MetaMathQA-style same-source hidden tests.
# The primary checkpoint metric is valid.json. Training uses all cleaned train rows;
# overlap-valid is kept only for diagnostics and is not used for epoch selection.
USE_KD = False
REQUIRE_KD_FILE = False

# Run mode: "phase1" reports overlap-valid + valid.json; "phase2" writes test_predictions.json.
RUN_MODE = "phase1"

# Type-aware prompt. `type` is a useful discriminator because the same original
# question can appear in SV/FOBAR/AnsAug variants with conflicting answers.
PROMPT_TEMPLATE = "Dạng: {type}\nBài toán: {q}\nLời giải: "
SAFE_EOS_ID = 50256
N_POSITIONS = 1024

# Working dirs / outputs
WORKING_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
STAGE_A_OUTPUT_DIR = WORKING_DIR / "gpt2_math_lora_v145_gate_sweep_valid_select_answer_only"
SFT_OUTPUT_DIR     = WORKING_DIR / "gpt2_math_lora_v145_gate_sweep_valid_select_sft"
FINAL_OUTPUT_DIR   = WORKING_DIR / "gpt2_math_lora_v145_gate_sweep_valid_select_final"

OVERLAP_VALID_OUTPUT_PATH       = WORKING_DIR / "overlap_valid_output.json"
OVERLAP_VALID_REPORT_PATH       = WORKING_DIR / "overlap_valid_report.json"
MODEL_OVERLAP_VALID_OUTPUT_PATH = WORKING_DIR / "model_overlap_valid_output.json"
MODEL_OVERLAP_VALID_REPORT_PATH = WORKING_DIR / "model_overlap_valid_report.json"
VALID_OUTPUT_PATH               = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH               = WORKING_DIR / "valid_report.json"
MODEL_VALID_OUTPUT_PATH         = WORKING_DIR / "model_valid_output.json"
MODEL_VALID_REPORT_PATH         = WORKING_DIR / "model_valid_report.json"
VALID_OVERLAP_AUDIT_PATH        = WORKING_DIR / "valid_overlap_audit.json"
QUERY_DISJOINT_SPLIT_PATH       = WORKING_DIR / "query_disjoint_split_report.json"
HYBRID_DECISION_REPORT_PATH     = WORKING_DIR / "hybrid_decision_report.json"
RETRIEVAL_GATE_SWEEP_REPORT_PATH = WORKING_DIR / "retrieval_gate_sweep_report.json"
SELECTED_RETRIEVAL_GATE_CONFIG_PATH = WORKING_DIR / "selected_retrieval_gate_config.json"
RL_LOG_PATH                     = WORKING_DIR / "rl_training_log.jsonl"
RL_SUMMARY_PATH                 = WORKING_DIR / "rl_reward_summary.json"
TEST_OUTPUT_PATH                = WORKING_DIR / "test_predictions.json"
MODEL_TEST_OUTPUT_PATH          = WORKING_DIR / "model_test_predictions.json"

# Checkpoint selection: source-overlap, exact-query-disjoint train-heldout.
CHECKPOINT_ROOT_DIR              = WORKING_DIR / "gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints"
CHECKPOINT_EVAL_DIR              = WORKING_DIR / "v145_gate_sweep_valid_select_checkpoint_eval_valid_json"
CHECKPOINT_SELECTION_REPORT_PATH = WORKING_DIR / "checkpoint_selection_report.json"
SELECTED_CHECKPOINT_INFO_PATH    = WORKING_DIR / "selected_checkpoint_info.json"
SELECT_CHECKPOINTS_ON_OVERLAP_VALID = True  # kept True so epoch adapters are saved; selection split is valid.json
CHECKPOINT_SELECTION_SPLIT_NAME = "valid_json"
OVERLAP_VALID_QUERY_FRACTION     = 0.10
OVERLAP_VALID_MAX_EVAL_RECORDS   = 1000
SOURCE_GROUP_KEY_FIELDS          = ["original_question_en", "original_question_vi", "query_vi"]
CHECKPOINT_EVAL_NUM_BEAMS        = 2
CHECKPOINT_EVAL_MAX_NEW_TOKENS   = 32
KEEP_CHECKPOINT_EVAL_OUTPUTS     = True
CHECKPOINT_TIE_BREAK             = "earlier_epoch"  # earlier_epoch or later_epoch

# V12/V13 hybrid retrieval. V11 leaves this disabled.
USE_HYBRID_RETRIEVAL = True
RETRIEVAL_STRATEGY = "source_type_majority"  # source_type_majority or source_nearest_query
RETRIEVAL_USE_FULL_TRAIN_FOR_REFERENCE_VALID = True
RETRIEVAL_MIN_SOURCE_CANDIDATES = 1
RETRIEVAL_MIN_MAJORITY_FRAC = 0.34  # fallback to model when same-source answers are too fragmented
RETRIEVAL_ALLOWED_TYPES = ["GSM_Rephrased", "MATH_Rephrased", "GSM_AnsAug", "MATH_AnsAug"]  # V13 gate; None means no type gate
RETRIEVAL_DEBUG_SAMPLE = 20

# V14 confidence gate. When the model already agrees with the retrieval
# candidate, we allow retrieval; when it disagrees, the candidate must
# pass stricter type-specific confidence thresholds.
RETRIEVAL_MIN_MAJORITY_MARGIN = 0.0
RETRIEVAL_MIN_NEAREST_JACCARD = 0.0
RETRIEVAL_USE_MODEL_AGREEMENT_GATE = True
RETRIEVAL_MODEL_AGREEMENT_BYPASS_CONFIDENCE = True
RETRIEVAL_TYPE_GATE_CONFIG = {}
ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG = {}
RETRIEVAL_GATE_SWEEP_ENABLED = True
RETRIEVAL_GATE_SWEEP_TYPES = ["GSM_Rephrased", "MATH_Rephrased", "GSM_AnsAug", "MATH_AnsAug"]
RETRIEVAL_GATE_SWEEP_GRID = {
    "_default": {
        "strategy": ["source_type_majority"],
        "min_majority_frac": [0.34, 0.50, 0.67, 0.75],
        "min_margin": [0.0, 0.15, 0.33, 0.50],
        "min_nearest_jaccard": [0.0, 0.45, 0.65, 0.80],
    },
    "GSM_AnsAug": {"min_majority_frac": [0.34, 0.45, 0.50, 0.67], "min_margin": [0.0, 0.10, 0.20, 0.33], "min_nearest_jaccard": [0.0, 0.35, 0.50, 0.65]},
    "MATH_AnsAug": {"min_majority_frac": [0.34, 0.45, 0.50, 0.67], "min_margin": [0.0, 0.10, 0.20, 0.33], "min_nearest_jaccard": [0.0, 0.35, 0.50, 0.65]},
}

# V14.5: train checkpoints directly on all cleaned train rows, then choose
# the checkpoint on valid.json. No post-selection full-train retrain is needed.
FINAL_RETRAIN_FULL_TRAIN = False
FINAL_RETRAIN_STAGE_NAME = "stage_a_full_train_final"
FULL_TRAIN_OUTPUT_DIR = WORKING_DIR / "gpt2_math_lora_v145_gate_sweep_valid_select_unused_final_retrain"
FINAL_RETRAIN_INFO_PATH = WORKING_DIR / "final_retrain_info.json"

# Smoke/debug knobs. For a fast local smoke run, set these small.
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
DROP_EXACT_DUPLICATES = True
DROP_NON_EXTRACTABLE = True

# V8-best style answer-only LoRA
STAGE_A_NAME = "stage_a_answer_only_lora"
STAGE_A_EPOCHS = 8.0
STAGE_A_LR = 1e-3
MAX_LENGTH_STAGE_A = 256

# Backward-compatible no-op stage variables for manifest cells.
RUN_STAGE_B = False
STAGE_B_EPOCHS = 0.0
STAGE_B_LR = 0.0
MAX_LENGTH_STAGE_B = 256
RUN_STAGE_C = False
STAGE_C_EPOCHS = 0.0
STAGE_C_LR = 0.0

# Trainer
PER_DEVICE_BATCH_SIZE = 16  # fallback: 8 if OOM
GRAD_ACCUM = 2              # fallback: 4 if PER_DEVICE_BATCH_SIZE=8
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01
SEED = 42

# LoRA
LORA_R = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["c_attn", "c_proj", "c_fc"]

# RL disabled.
RL_ENABLED = False
RL_MAX_STEPS = 0

# Generation defaults for evaluation/submission
MAX_NEW_TOKENS = 32
NUM_BEAMS = 2
DECODE_BATCH_SIZE = 8
NO_REPEAT_NGRAM = 4
REPETITION_PENALTY = 1.15
LENGTH_PENALTY = 0.9
SANITIZE_TO_ANSWER_ONLY = True
INFER_FP16 = True

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("TRAIN_FILE       :", TRAIN_FILE)
print("VALID_FILE       :", VALID_FILE)
print("TEST_FILE        :", TEST_FILE, "| exists:", TEST_FILE.exists())
print("MODEL_NAME       :", MODEL_NAME)
print("RUN_MODE         :", RUN_MODE)
print("USE_HYBRID       :", USE_HYBRID_RETRIEVAL)
print("RETRIEVAL_GATE   :", RETRIEVAL_ALLOWED_TYPES)
print("FINAL_RETRAIN    :", FINAL_RETRAIN_FULL_TRAIN)
print("FINAL_OUTPUT_DIR :", FINAL_OUTPUT_DIR)
print("CHECKPOINT_ROOT  :", CHECKPOINT_ROOT_DIR)


TRAIN_FILE       : /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_FILE       : /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
TEST_FILE        : /kaggle/input/datasets/kimanh2002/dataset-math/test.json | exists: False
MODEL_NAME       : /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
RUN_MODE         : phase1
USE_HYBRID       : True
RETRIEVAL_GATE   : ['GSM_Rephrased', 'MATH_Rephrased', 'GSM_AnsAug', 'MATH_AnsAug']
FINAL_RETRAIN    : False
FINAL_OUTPUT_DIR : /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_final
CHECKPOINT_ROOT  : /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints


In [3]:
# ============================================================
# 2. Data loading + robust numeric evaluator
# ============================================================
def load_records(path: str | Path) -> list[dict]:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

ANSWER_ANCHORS = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}
EOS_ARTIFACT_RE = re.compile(r"(?:\s*(?:hue|<\|endoftext\|>|</s>|<pad>))+\s*$", re.IGNORECASE)

def clean_decoded_artifacts(text: str | None) -> str:
    """Remove decoded pseudo-EOS artifacts before answer parsing/saving.

    The task asks us to use SAFE_EOS_ID=50256 because the GPT-2 Vietnamese model
    embedding matrix is sized for ids 0..50256. In this tokenizer, however,
    id 50256 decodes to the ordinary string "hue", not to a special token.
    If generation stops on this id, Hugging Face includes it in decoded text.
    Official scoring needs a clean numeric answer, so strip only trailing
    terminator artifacts.
    """
    text = str(text or "").strip()
    for _ in range(4):
        new_text = EOS_ARTIFACT_RE.sub("", text).strip()
        if new_text == text:
            break
        text = new_text
    return text

def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # V3-stable behavior: id 50256 is required as model EOS/PAD but decodes to
    # the ordinary token "hue" in this tokenizer, so strip it by id before
    # decoding rather than relying on skip_special_tokens.
    return clean_decoded_artifacts(tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True))

def _clean_tail(text: str) -> str:
    text = clean_decoded_artifacts(text).split("\n", 1)[0].strip()
    text = re.sub(r"[.,;:。、“”\"')\]]+$", "", text).strip()
    text = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)\b.*$", "", text, flags=re.IGNORECASE)
    return text.strip()

def extract_answer(text: str | None) -> str | None:
    if not text:
        return None
    best_end = -1
    best_tail = None
    for pat in ANSWER_ANCHORS:
        for m in pat.finditer(text):
            if m.end() > best_end:
                best_end = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    boxes = BOXED_RE.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(text: str | None) -> float | None:
    if text is None:
        return None
    value = str(text).strip()
    if not value:
        return None
    if re.fullmatch(r"-?\d+,\d+", value):
        try:
            return float(value.replace(",", "."))
        except ValueError:
            return None
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", value):
        try:
            parsed = float(value)
            return parsed if math.isfinite(parsed) else None
        except ValueError:
            return None
    assignment = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", value)
    if assignment:
        value = assignment.group(1).strip()
    if value.startswith("(") and value.endswith(")") and re.search(r"\d\s*,\s*\d", value):
        return None
    if value.startswith("[") and value.endswith("]"):
        return None
    for _ in range(3):
        new_value = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", value)
        if new_value == value:
            break
        value = new_value
    value = re.sub(r"\\text\{[^}]*\}", "", value)
    value = re.sub(r"\\mathrm\{[^}]*\}", "", value)
    value = value.replace("$", "")
    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        value = value.replace(token, "")
    for token in ("\\cdot", "\\times"):
        value = value.replace(token, "*")
    value = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", value)
    value = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", value)
    value = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", value)
    value = value.replace("\\pi", "pi")
    value = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", value)
    value = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", value)
    value = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", value)
    has_period = "." in value
    comma_count = value.count(",")
    if comma_count == 1 and not has_period and re.search(r"\d,\d", value):
        value = re.sub(r"(?<=\d),(?=\d)", ".", value)
    elif comma_count >= 1:
        value = re.sub(r"(?<=\d),(?=\d{3}\b)", "", value)
    value = re.sub(r"\s+", "", value)
    if not value or "," in value:
        return None
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", value)
    if leftover:
        return None
    try:
        parsed = eval(value.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None
    if isinstance(parsed, bool):
        return None
    if isinstance(parsed, (int, float)):
        parsed = float(parsed)
        return parsed if math.isfinite(parsed) else None
    return None

def extract_gold(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("response_vi"))
    return answer, parse_number(answer)

def extract_pred(record: dict) -> tuple[str | None, float | None]:
    answer = extract_answer(record.get("model_output"))
    return answer, parse_number(answer)

def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(error_value: float | None, extractable: bool) -> int:
    if not extractable or error_value is None:
        return 0
    if error_value <= 0.01:
        return 10
    if error_value <= 0.10:
        return 5
    if error_value <= 0.50:
        return 1
    return 0

def evaluate_predictions(pred_items: list[dict], gold_items: list[dict]) -> dict:
    if len(pred_items) != len(gold_items):
        raise ValueError(f"Prediction count {len(pred_items)} != gold count {len(gold_items)}")
    rows, total, extractable, numeric_pairs, rel_errors = [], 0, 0, 0, []
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    by_type = defaultdict(lambda: {"n": 0, "raw_score": 0, "extractable": 0, "bucket_10": 0, "bucket_5": 0, "bucket_1": 0, "bucket_0": 0})
    for pred, gold in zip(pred_items, gold_items):
        gold_answer, gold_num = extract_gold(gold)
        pred_answer, pred_num = extract_pred(pred)
        is_extractable = pred_answer is not None
        error_value = rel_error(pred_num, gold_num)
        score = score_one(error_value, is_extractable)
        t = gold.get("type") or pred.get("type") or "unknown"
        total += score
        extractable += int(is_extractable)
        buckets[score] = buckets.get(score, 0) + 1
        if gold_num is not None and pred_num is not None and error_value is not None:
            numeric_pairs += 1
            rel_errors.append(error_value)
        by_type[t]["n"] += 1
        by_type[t]["raw_score"] += score
        by_type[t]["extractable"] += int(is_extractable)
        by_type[t][f"bucket_{score}"] += 1
        rows.append({
            "id": gold.get("id", pred.get("id")),
            "type": t,
            "gold_answer": gold_answer,
            "gold_num": gold_num,
            "pred_answer": pred_answer,
            "pred_num": pred_num,
            "rel_error": error_value,
            "extractable": is_extractable,
            "score": score,
        })
    n = len(rows)
    by_type_final = {}
    for t, d in sorted(by_type.items()):
        d = dict(d)
        d["score_10"] = d["raw_score"] / d["n"] if d["n"] else 0.0
        by_type_final[t] = d
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": 10 * n,
            "score_10": total / n if n else 0.0,
            "score_pct": total / (10 * n) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "by_type": by_type_final,
        "rows": rows,
    }

def save_eval_report(pred_path: Path, gold_records: list[dict], report_path: Path) -> dict:
    pred_items = json.loads(Path(pred_path).read_text(encoding="utf-8"))
    report = evaluate_predictions(pred_items, gold_records)
    Path(report_path).write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    return report

def canonicalize_answer(gold_num: float | None) -> str | None:
    if gold_num is None:
        return None
    if abs(gold_num - round(gold_num)) < 1e-9:
        return str(int(round(gold_num)))
    return f"{gold_num:g}"

def sanitize_model_output(text: str | None) -> str:
    """Save a clean answer line if the model produced a parseable answer.

    This is prediction post-processing only: it uses the model's own decoded
    answer, never the gold answer. It prevents harmless decoded terminators or
    extra continuation text from making an otherwise numeric prediction
    unparseable by the official-style scorer.
    """
    cleaned = clean_decoded_artifacts(text)
    pred_answer = extract_answer(cleaned)
    pred_num = parse_number(pred_answer)
    canonical = canonicalize_answer(pred_num)
    if canonical is not None:
        return f"Đáp án là: {canonical}"
    return cleaned

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
test_records_for_info = load_records(TEST_FILE) if TEST_FILE.exists() else []

if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records), "| test:", len(test_records_for_info))
print("train type distribution:", dict(Counter(r.get("type") for r in train_records).most_common()))


train: 95400 | valid: 1000 | test: 0
train type distribution: {'GSM_Rephrased': 20028, 'GSM_AnsAug': 18745, 'MATH_AnsAug': 16999, 'MATH_Rephrased': 12477, 'GSM_FOBAR': 10023, 'GSM_SV': 9869, 'MATH_FOBAR': 3668, 'MATH_SV': 3591}


In [4]:
# ============================================================
# 3. Clean data + source-overlap query-disjoint split
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID
tokenizer.padding_side = "left"
tokenizer.truncation_side = "left"

def stable_fraction(value) -> float:
    h = hashlib.sha256(str(value).encode("utf-8")).hexdigest()
    return int(h[:8], 16) / 0x100000000

def normalize_text_key(text: str | None) -> str:
    text = unicodedata.normalize("NFKC", str(text or "")).lower()
    text = re.sub(r"\s+", " ", text).strip()
    return text

def source_group_key(rec: dict) -> str:
    for field in SOURCE_GROUP_KEY_FIELDS:
        value = normalize_text_key(rec.get(field))
        if value:
            return f"{field}:{value}"
    return f"source_id:{rec.get('_source_id', rec.get('id', 'unknown'))}"

def query_key(rec: dict) -> str:
    value = normalize_text_key(rec.get("query_vi"))
    return value or f"query_id:{rec.get('_source_id', rec.get('id', 'unknown'))}"

def query_tokens(text: str | None) -> set[str]:
    return set(re.findall(r"\w+", normalize_text_key(text)))

def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def format_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(type=rec.get("type") or "unknown", q=(rec.get("query_vi") or "").strip())

def build_answer_only_target(canonical_answer: str) -> str:
    return f"Đáp án là: {canonical_answer}"

def save_valid_overlap_audit(train_recs: list[dict], valid_recs: list[dict], path: Path):
    train_q = Counter(query_key(r) for r in train_recs)
    train_source = Counter(source_group_key(r) for r in train_recs)
    seen_query = []
    seen_source = []
    by_type = defaultdict(lambda: {"n": 0, "seen_query": 0, "seen_source": 0})
    for i, rec in enumerate(valid_recs):
        qk = query_key(rec)
        sk = source_group_key(rec)
        t = rec.get("type") or "unknown"
        by_type[t]["n"] += 1
        if qk in train_q:
            seen_query.append(i)
            by_type[t]["seen_query"] += 1
        if sk in train_source:
            seen_source.append(i)
            by_type[t]["seen_source"] += 1
    report = {
        "train_n": len(train_recs),
        "valid_n": len(valid_recs),
        "train_unique_query": len(train_q),
        "train_unique_source_group": len(train_source),
        "valid_seen_query": len(seen_query),
        "valid_seen_query_pct": len(seen_query) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_source_group": len(seen_source),
        "valid_seen_source_group_pct": len(seen_source) / len(valid_recs) if valid_recs else 0.0,
        "valid_seen_query_ids_first20": seen_query[:20],
        "valid_seen_source_group_ids_first20": seen_source[:20],
        "valid_by_type": dict(sorted((k, dict(v)) for k, v in by_type.items())),
    }
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[valid-overlap]", json.dumps({k: report[k] for k in ["train_n", "valid_n", "valid_seen_query", "valid_seen_source_group"]}, ensure_ascii=False))
    return report

def clean_records(records: list[dict], split: str) -> list[dict]:
    seen_exact = set()
    out = []
    dropped_dup = dropped_empty = dropped_non_numeric = 0
    for source_id, rec in enumerate(records):
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_empty += 1
            continue
        if split == "train" and DROP_EXACT_DUPLICATES:
            key = (q, r)
            if key in seen_exact:
                dropped_dup += 1
                continue
            seen_exact.add(key)
        gold_str, gold_num = extract_gold(rec)
        canonical = canonicalize_answer(gold_num)
        if DROP_NON_EXTRACTABLE and canonical is None:
            dropped_non_numeric += 1
            continue
        out.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
            "_source_id": source_id,
            "_source_key": None,  # filled below after helpers exist
            "_query_key": None,
            "_query_tokens": None,
            "_gold_str": gold_str,
            "_gold_num": gold_num,
            "_canonical_answer": canonical,
        })
    for rec in out:
        rec["_source_key"] = source_group_key(rec)
        rec["_query_key"] = query_key(rec)
        rec["_query_tokens"] = query_tokens(rec.get("query_vi"))
    print(f"[clean:{split}] kept={len(out)} dropped_dup={dropped_dup} dropped_empty={dropped_empty} dropped_non_numeric={dropped_non_numeric}")
    return out

def split_source_overlap_query_disjoint(records: list[dict]) -> tuple[list[dict], list[dict], dict]:
    """Hold out query variants while keeping each heldout source in train.

    This simulates MetaMathQA hidden examples that share an original question
    with train but use a different augmented query.
    """
    by_source_query = defaultdict(lambda: defaultdict(list))
    for rec in records:
        by_source_query[rec["_source_key"]][rec["_query_key"]].append(rec)

    rng = random.Random(SEED)
    valid_query_pairs = set()
    source_query_counts = {}
    skipped_single_query_groups = 0
    for sk, qmap in by_source_query.items():
        qkeys = sorted(qmap)
        source_query_counts[sk] = len(qkeys)
        if len(qkeys) < 2:
            skipped_single_query_groups += 1
            continue
        shuffled = qkeys[:]
        rng.shuffle(shuffled)
        n_valid = max(1, int(round(len(shuffled) * OVERLAP_VALID_QUERY_FRACTION)))
        n_valid = min(n_valid, len(shuffled) - 1)
        for qk in shuffled[:n_valid]:
            valid_query_pairs.add((sk, qk))

    train_out, valid_out = [], []
    for rec in records:
        pair = (rec["_source_key"], rec["_query_key"])
        if pair in valid_query_pairs:
            valid_out.append(rec)
        else:
            train_out.append(rec)

    train_sources = {r["_source_key"] for r in train_out}
    valid_sources = {r["_source_key"] for r in valid_out}
    train_queries = {(r["_source_key"], r["_query_key"]) for r in train_out}
    valid_queries = {(r["_source_key"], r["_query_key"]) for r in valid_out}
    report = {
        "split_name": "source_overlap_query_disjoint",
        "query_fraction": OVERLAP_VALID_QUERY_FRACTION,
        "total_records": len(records),
        "total_source_groups": len(by_source_query),
        "skipped_single_query_source_groups": skipped_single_query_groups,
        "eligible_source_groups": len(by_source_query) - skipped_single_query_groups,
        "train_records": len(train_out),
        "overlap_valid_records": len(valid_out),
        "train_source_groups": len(train_sources),
        "overlap_valid_source_groups": len(valid_sources),
        "source_group_overlap": len(train_sources & valid_sources),
        "query_pair_overlap": len(train_queries & valid_queries),
        "overlap_valid_by_type": dict(Counter(r.get("type") or "unknown" for r in valid_out).most_common()),
        "train_by_type": dict(Counter(r.get("type") or "unknown" for r in train_out).most_common()),
        "source_query_count_hist": dict(Counter(source_query_counts.values()).most_common()),
    }
    if report["query_pair_overlap"] != 0:
        raise RuntimeError(f"query-disjoint split failed: {report['query_pair_overlap']} query pairs overlap")
    print("[query-disjoint-split]", json.dumps({k: report[k] for k in ["train_records", "overlap_valid_records", "source_group_overlap", "query_pair_overlap"]}, ensure_ascii=False))
    return train_out, valid_out, report

def build_balanced_subset(records: list[dict], max_records: int | None, seed: int) -> list[dict]:
    if max_records is None or max_records >= len(records):
        return list(records)
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for rec in records:
        buckets[rec.get("type") or "unknown"].append(rec)
    for vals in buckets.values():
        rng.shuffle(vals)
    active_types = sorted(buckets)
    out = []
    cursor = 0
    while len(out) < max_records and active_types:
        t = active_types[cursor % len(active_types)]
        if buckets[t]:
            out.append(buckets[t].pop())
        if not buckets[t]:
            active_types.remove(t)
            cursor = 0
        else:
            cursor += 1
    rng.shuffle(out)
    return out

def build_answer_only_records(records: list[dict], stage_name: str) -> list[dict]:
    out = []
    for rec in records:
        canonical = rec.get("_canonical_answer")
        if canonical is not None:
            out.append({**rec, "response_vi": build_answer_only_target(canonical), "_stage": stage_name})
    print(f"[build:{stage_name}] {len(out)}")
    return out

valid_overlap_audit = save_valid_overlap_audit(train_records, valid_records, VALID_OVERLAP_AUDIT_PATH)
train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid_reference")
# Valid-select runs intentionally skip the overlap-valid holdout.
overlap_train_clean = train_clean
overlap_valid_clean = []
overlap_valid_eval_records = []
query_disjoint_split_report = {
    "split_name": "skipped_valid_select_full_train",
    "reason": "train on all cleaned train rows and select checkpoint on valid.json",
    "total_records": len(train_clean),
    "train_records": len(train_clean),
    "overlap_valid_records": 0,
    "overlap_valid_eval_n": 0,
}
QUERY_DISJOINT_SPLIT_PATH.write_text(json.dumps(query_disjoint_split_report, ensure_ascii=False, indent=2), encoding="utf-8")

train_stage_a = build_answer_only_records(train_clean, STAGE_A_NAME)
full_train_stage_a = build_answer_only_records(train_clean, FINAL_RETRAIN_STAGE_NAME)
overlap_valid_stage_a = build_answer_only_records(overlap_valid_eval_records, STAGE_A_NAME + "_overlap_valid")

print("\nExample target:")
print(train_stage_a[0]["response_vi"])
print("Example prompt:")
print(format_prompt(train_stage_a[0]))


[valid-overlap] {"train_n": 95400, "valid_n": 1000, "valid_seen_query": 0, "valid_seen_source_group": 965}
[clean:train] kept=92874 dropped_dup=0 dropped_empty=0 dropped_non_numeric=2526
[clean:valid_reference] kept=977 dropped_dup=0 dropped_empty=0 dropped_non_numeric=23
[build:stage_a_answer_only_lora] 92874
[build:stage_a_full_train_final] 92874
[build:stage_a_answer_only_lora_overlap_valid] 0

Example target:
Đáp án là: 1200
Example prompt:
Dạng: GSM_AnsAug
Bài toán: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn chuẩn bị thêm mười đĩa đề phòng trường hợp có sự cố xảy ra. Mỗi đĩa bít tết và măng tây sốt bơ tỏi sẽ có 8 ngọn măng tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?
Lời giải: 


In [5]:
# ============================================================
# 4. SFT dataset: pre-tokenized, loss only on response tokens
# ============================================================
class SFTDataset(Dataset):
    """Pre-tokenize once instead of tokenizing inside __getitem__."""
    def __init__(self, records, tokenizer, max_length: int, desc: str = "train"):
        self.examples = []
        self.tok = tokenizer
        self.max_length = max_length
        for rec in tqdm(records, desc=f"tokenize:{desc}", leave=False):
            prompt = format_prompt(rec)
            response = rec["response_vi"]
            p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
            r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

            if len(p_ids) >= self.max_length - 1:
                p_ids = p_ids[-(self.max_length - 1):]
            budget = self.max_length - len(p_ids)
            if budget <= 0:
                r_ids = [SAFE_EOS_ID]
            elif len(r_ids) > budget:
                r_ids = r_ids[-budget:]

            ids = p_ids + r_ids
            labels = [-100] * len(p_ids) + r_ids
            ids = [min(t, SAFE_EOS_ID) for t in ids]
            labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]
            self.examples.append({
                "input_ids": ids,
                "attention_mask": [1] * len(ids),
                "labels": labels,
                "length": len(ids),
            })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

@dataclass
class PadCollator:
    pad_id: int
    pad_to_multiple_of: int = 8

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        if self.pad_to_multiple_of:
            m = self.pad_to_multiple_of
            maxlen = ((maxlen + m - 1) // m) * m
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

_probe = SFTDataset(train_stage_a[:1], tokenizer, MAX_LENGTH_STAGE_A, desc="probe")[0]
print("stage_a len/loss_tokens:", len(_probe["input_ids"]), sum(x != -100 for x in _probe["labels"]))
print("stage_a tail:", decode_model_text(tokenizer, _probe["input_ids"][-30:]))


tokenize:probe:   0%|          | 0/1 [00:00<?, ?it/s]

stage_a len/loss_tokens: 125 6
stage_a tail: tây trên đó. Người cung cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?
Lời giải: Đáp án là: 1200


In [6]:
# ============================================================
# 5. PEFT LoRA SFT: answer-only with per-epoch checkpoints
# ============================================================
def build_training_args(output_dir: Path, epochs: float, lr: float):
    kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        save_strategy="no",
        report_to="none",
        seed=SEED,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        remove_unused_columns=False,
        group_by_length=True,
    )
    sig = inspect.signature(TrainingArguments.__init__)
    if "eval_strategy" in sig.parameters:
        kwargs["eval_strategy"] = "no"
    else:
        kwargs["evaluation_strategy"] = "no"
    if "optim" in sig.parameters and torch.cuda.is_available():
        kwargs["optim"] = "adamw_torch_fused"
    return TrainingArguments(**kwargs)

def build_lora_model():
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = False
    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model

CHECKPOINT_SAVED = []

def _checkpoint_label(epoch_value: float | None, *, final: bool = False) -> str:
    if final:
        return f"final_epoch_{float(STAGE_A_EPOCHS):.2f}".replace(".", "p")
    if epoch_value is None:
        return f"epoch_unknown_{len(CHECKPOINT_SAVED)+1:02d}"
    ev = float(epoch_value)
    if abs(ev - round(ev)) < 1e-3:
        return f"epoch_{int(round(ev)):02d}"
    return f"epoch_{ev:.2f}".replace(".", "p")

def save_adapter_checkpoint(model, root_dir: Path, *, epoch_value: float | None, final: bool = False, stage_name: str = STAGE_A_NAME):
    root_dir.mkdir(parents=True, exist_ok=True)
    label = _checkpoint_label(epoch_value, final=final)
    ckpt_dir = root_dir / label
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(ckpt_dir)
    tokenizer.save_pretrained(ckpt_dir)
    meta = {
        "label": label,
        "stage_name": stage_name,
        "epoch": None if epoch_value is None else float(epoch_value),
        "final": bool(final),
        "stage_a_epochs": STAGE_A_EPOCHS,
        "stage_a_lr": STAGE_A_LR,
        "prompt_template": PROMPT_TEMPLATE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "max_length_stage_a": MAX_LENGTH_STAGE_A,
        "saved_at_unix": time.time(),
    }
    (ckpt_dir / "checkpoint_meta.json").write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    try:
        meta["sha256"] = sha256_dir(ckpt_dir)
        (ckpt_dir / "model_hash.txt").write_text(meta["sha256"] + "\n", encoding="utf-8")
    except Exception as exc:
        meta["sha256_error"] = repr(exc)
    CHECKPOINT_SAVED.append(meta | {"path": str(ckpt_dir)})
    print(f"[checkpoint] saved {label} -> {ckpt_dir}")
    return ckpt_dir

class SaveAdapterEveryEpochCallback(TrainerCallback):
    def __init__(self, root_dir: Path):
        self.root_dir = Path(root_dir)
        self._saved_labels = set()

    def on_epoch_end(self, args, state, control, **kwargs):
        model_obj = kwargs.get("model")
        if model_obj is None or state.epoch is None:
            return control
        label = _checkpoint_label(float(state.epoch))
        if label in self._saved_labels:
            return control
        save_adapter_checkpoint(model_obj, self.root_dir, epoch_value=float(state.epoch), final=False)
        self._saved_labels.add(label)
        return control

def train_lora_stage(
    model,
    train_records_for_stage,
    output_dir: Path,
    max_length: int,
    epochs: float,
    lr: float,
    *,
    stage_name: str = STAGE_A_NAME,
    save_epoch_checkpoints: bool = True,
):
    print("\n" + "=" * 90)
    print(f"[train:{stage_name}] train={len(train_records_for_stage)} max_length={max_length} epochs={epochs} lr={lr}")
    train_ds = SFTDataset(train_records_for_stage, tokenizer, max_length, desc=stage_name)
    collator = PadCollator(SAFE_EOS_ID)
    eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
    print(f"[train:{stage_name}] eff_batch={eff_batch} steps/epoch={math.ceil(len(train_ds)/eff_batch)}")

    callbacks = []
    if SELECT_CHECKPOINTS_ON_OVERLAP_VALID and save_epoch_checkpoints:
        callbacks.append(SaveAdapterEveryEpochCallback(CHECKPOINT_ROOT_DIR))

    trainer = Trainer(
        model=model,
        args=build_training_args(output_dir, epochs, lr),
        train_dataset=train_ds,
        data_collator=collator,
        callbacks=callbacks,
    )
    t0 = time.time()
    trainer.train()
    dt = time.time() - t0
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    model_hash = sha256_dir(output_dir)
    (output_dir / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    print(f"[train:{stage_name}] wall={dt/60:.2f} min saved={output_dir} sha256={model_hash}")
    if SELECT_CHECKPOINTS_ON_OVERLAP_VALID and save_epoch_checkpoints:
        epoch_float = float(epochs)
        integer_epoch_dir = CHECKPOINT_ROOT_DIR / _checkpoint_label(epoch_float)
        if abs(epoch_float - round(epoch_float)) < 1e-3 and (integer_epoch_dir / "adapter_config.json").exists():
            print(f"[checkpoint] final epoch duplicates {integer_epoch_dir.name}; skip extra final checkpoint")
        else:
            save_adapter_checkpoint(model, CHECKPOINT_ROOT_DIR, epoch_value=epoch_float, final=True, stage_name=stage_name)
        (CHECKPOINT_ROOT_DIR / "checkpoint_index.json").write_text(json.dumps(CHECKPOINT_SAVED, ensure_ascii=False, indent=2), encoding="utf-8")
    del trainer
    torch.cuda.empty_cache()
    return model, dt

model = build_lora_model()
model, stage_a_train_dt = train_lora_stage(
    model,
    train_stage_a,
    STAGE_A_OUTPUT_DIR,
    MAX_LENGTH_STAGE_A,
    STAGE_A_EPOCHS,
    STAGE_A_LR,
)

SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
(SFT_OUTPUT_DIR / "model_hash.txt").write_text(sha256_dir(SFT_OUTPUT_DIR) + "\n", encoding="utf-8")

FINAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(FINAL_OUTPUT_DIR)
tokenizer.save_pretrained(FINAL_OUTPUT_DIR)
(FINAL_OUTPUT_DIR / "model_hash.txt").write_text(sha256_dir(FINAL_OUTPUT_DIR) + "\n", encoding="utf-8")

print(f"[train:sft] total wall={stage_a_train_dt/60:.2f} min")


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 4,718,592 || all params: 129,158,400 || trainable%: 3.6533

[train:stage_a_answer_only_lora] train=92874 max_length=256 epochs=8.0 lr=0.001


tokenize:stage_a_answer_only_lora:   0%|          | 0/92874 [00:00<?, ?it/s]

[train:stage_a_answer_only_lora] eff_batch=64 steps/epoch=1452


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
50,5.673980
100,1.827621
150,0.938823
200,0.915768
250,0.895058
300,0.880780
350,0.863833
400,0.848556
450,0.843670
500,0.840468


[checkpoint] saved epoch_01 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_01
[checkpoint] saved epoch_02 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_02
[checkpoint] saved epoch_03 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_03
[checkpoint] saved epoch_04 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_04
[checkpoint] saved epoch_05 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_05
[checkpoint] saved epoch_06 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_06
[checkpoint] saved epoch_07 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_07
[checkpoint] saved epoch_08 -> /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_08
[train:stage_a_answer_only_lora] wall=175.51 min saved=/kaggle/working/gpt2_math_lora_v145_gate_sweep_va

In [7]:
# ============================================================
# 6. Prompt helper / RL disabled
# ============================================================
def build_prompt(rec: dict) -> str:
    return format_prompt(rec)

rl_summary = {
    "enabled": False,
    "reason": "This notebook is SFT-only; no GRPO/RL stage.",
    "final_dir": str(FINAL_OUTPUT_DIR),
}
RL_SUMMARY_PATH.write_text(json.dumps(rl_summary, ensure_ascii=False, indent=2), encoding="utf-8")

del model
torch.cuda.empty_cache()


In [8]:
# ============================================================
# 7. Generation + overlap-valid checkpoint selection
# ============================================================
def has_peft_adapter(path_like) -> bool:
    p = Path(path_like)
    return p.exists() and (p / "adapter_config.json").exists()

def load_model_for_generation(adapter_dir: Path):
    dtype = torch.float16 if (INFER_FP16 and torch.cuda.is_available()) else torch.float32
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=dtype, local_files_only=True)
    base.config.pad_token_id = SAFE_EOS_ID
    base.config.eos_token_id = SAFE_EOS_ID
    if has_peft_adapter(adapter_dir):
        print(f"[infer] base + adapter: {adapter_dir}")
        gen_model = PeftModel.from_pretrained(base, str(adapter_dir), local_files_only=True)
        try:
            gen_model = gen_model.merge_and_unload()
            print("[infer] merged LoRA adapter")
        except Exception as exc:
            print("[infer] merge failed; using PEFT wrapper:", repr(exc))
    else:
        print(f"[infer] no adapter at {adapter_dir}; using base model")
        gen_model = base
    device = "cuda" if torch.cuda.is_available() else "cpu"
    gen_model.to(device)
    gen_model.eval()
    return gen_model

class StopOnAnswerLine(StoppingCriteria):
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID, patience_tokens: int = 8):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_answer = re.compile(r"đáp\s*án\s*l[àa]\s*[:：]\s*-?\d", re.IGNORECASE)
        self._matched_at = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 4:
            return False
        text = decode_model_text(self.tok, gen_tail)
        m = self.re_answer.search(text)
        if not m:
            return False
        if self._matched_at is None:
            self._matched_at = gen_tail.numel()
        if "\n" in text[m.end():]:
            return True
        if gen_tail.numel() - self._matched_at >= self.patience:
            return True
        return False

@torch.inference_mode()
def generate_model_outputs(adapter_dir: Path, records: list[dict], output_path: Path, *, max_new_tokens: int = MAX_NEW_TOKENS, num_beams: int = NUM_BEAMS):
    gen_tok = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
    gen_tok.pad_token_id = SAFE_EOS_ID
    gen_tok.eos_token_id = SAFE_EOS_ID
    gen_tok.padding_side = "left"
    gen_tok.truncation_side = "left"
    gen_model = load_model_for_generation(adapter_dir)
    device = next(gen_model.parameters()).device
    outputs = []
    n_pos = int(getattr(gen_model.config, "n_positions", getattr(gen_model.config, "max_position_embeddings", 1024)))
    vocab_n = gen_model.get_input_embeddings().num_embeddings
    for idx, rec in enumerate(tqdm(records, desc=f"generate:{Path(adapter_dir).name}")):
        prompt = build_prompt(rec)
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = gen_tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        full_ids = [min(t, vocab_n - 1) for t in full_ids]
        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]
        eff_new = max(8, min(max_new_tokens, n_pos - prompt_len))
        gen_kwargs = dict(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            no_repeat_ngram_size=NO_REPEAT_NGRAM,
            stopping_criteria=StoppingCriteriaList([StopOnAnswerLine(gen_tok, prompt_len=prompt_len)]),
        )
        if num_beams and num_beams > 1:
            gen_kwargs.update(dict(num_beams=num_beams, do_sample=False, early_stopping=True, length_penalty=LENGTH_PENALTY))
        else:
            gen_kwargs.update(dict(num_beams=1, do_sample=False))
        seqs = gen_model.generate(**gen_kwargs)
        text = decode_model_text(gen_tok, seqs[0, prompt_len:])
        if SANITIZE_TO_ANSWER_ONLY:
            text = sanitize_model_output(text)
        outputs.append({
            "id": rec.get("id", idx),
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "model_output": text.strip(),
        })
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[infer] wrote {len(outputs)} model rows -> {output_path}")
    del gen_model
    torch.cuda.empty_cache()
    return outputs

def build_retrieval_index(records: list[dict]) -> dict:
    index = defaultdict(list)
    for rec in records:
        if rec.get("_canonical_answer") is None or rec.get("_gold_num") is None:
            continue
        index[rec["_source_key"]].append({
            "type": rec.get("type") or "unknown",
            "answer_num": float(rec["_gold_num"]),
            "canonical_answer": rec["_canonical_answer"],
            "query_tokens": rec.get("_query_tokens") or query_tokens(rec.get("query_vi")),
            "query_vi": rec.get("query_vi", ""),
        })
    return dict(index)

RETRIEVAL_INDEX_OVERLAP_TRAIN = build_retrieval_index(overlap_train_clean)
RETRIEVAL_INDEX_FULL_TRAIN = build_retrieval_index(train_clean)
print("[retrieval] overlap-train source groups:", len(RETRIEVAL_INDEX_OVERLAP_TRAIN))
print("[retrieval] full-train source groups:", len(RETRIEVAL_INDEX_FULL_TRAIN))

def _num_equal(a, b, tol: float = 1e-9) -> bool:
    if a is None or b is None:
        return False
    try:
        a = float(a)
        b = float(b)
    except Exception:
        return False
    return abs(a - b) <= tol * max(1.0, abs(a), abs(b))

def _model_num(model_item: dict | None):
    if not model_item:
        return None
    try:
        _, pred_num = extract_pred(model_item)
        return pred_num
    except Exception:
        return None

def retrieval_config_for_type(rec_type: str, gate_config_by_type: dict | None = None) -> dict:
    cfg = {
        "enabled": True,
        "strategy": RETRIEVAL_STRATEGY,
        "min_majority_frac": RETRIEVAL_MIN_MAJORITY_FRAC,
        "min_margin": RETRIEVAL_MIN_MAJORITY_MARGIN,
        "min_nearest_jaccard": RETRIEVAL_MIN_NEAREST_JACCARD,
        "require_typed_pool": False,
        "use_model_agreement_gate": RETRIEVAL_USE_MODEL_AGREEMENT_GATE,
        "model_agreement_bypass_confidence": RETRIEVAL_MODEL_AGREEMENT_BYPASS_CONFIDENCE,
    }
    for source in (RETRIEVAL_TYPE_GATE_CONFIG, ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG, gate_config_by_type):
        if source and rec_type in source:
            cfg.update(source[rec_type])
    return cfg

def _rank_answer_groups(pool: list[dict], rec_tokens: set[str]) -> list[dict]:
    by_num = defaultdict(list)
    for c in pool:
        by_num[c["answer_num"]].append(c)
    ranked = []
    for num, vals in by_num.items():
        nearest_j = max(jaccard(rec_tokens, v["query_tokens"]) for v in vals)
        ranked.append({
            "answer_num": num,
            "canonical_answer": vals[0]["canonical_answer"],
            "count": len(vals),
            "nearest_jaccard_same_answer": nearest_j,
        })
    ranked.sort(key=lambda x: (x["count"], x["nearest_jaccard_same_answer"], -abs(x["answer_num"])), reverse=True)
    return ranked

def _selected_from_majority(pool: list[dict], rec_tokens: set[str]) -> tuple[dict, list[dict]]:
    ranked = _rank_answer_groups(pool, rec_tokens)
    selected = dict(ranked[0])
    selected["selection_strategy"] = "source_type_majority"
    return selected, ranked

def _selected_from_nearest(pool: list[dict], rec_tokens: set[str]) -> tuple[dict, list[dict]]:
    best = max(pool, key=lambda c: jaccard(rec_tokens, c["query_tokens"]))
    ranked = _rank_answer_groups(pool, rec_tokens)
    selected = None
    for group in ranked:
        if _num_equal(group["answer_num"], best["answer_num"]):
            selected = dict(group)
            break
    if selected is None:
        selected = {
            "answer_num": best["answer_num"],
            "canonical_answer": best["canonical_answer"],
            "count": 1,
            "nearest_jaccard_same_answer": jaccard(rec_tokens, best["query_tokens"]),
        }
    selected["canonical_answer"] = best["canonical_answer"]
    selected["nearest_query_jaccard"] = jaccard(rec_tokens, best["query_tokens"])
    selected["selection_strategy"] = "source_type_nearest_query"
    return selected, ranked

def retrieve_answer_for_record(rec: dict, retrieval_index: dict, model_item: dict | None = None, gate_config_by_type: dict | None = None) -> dict:
    rec_type = rec.get("type") or "unknown"
    cfg = retrieval_config_for_type(rec_type, gate_config_by_type)
    candidates = retrieval_index.get(source_group_key(rec), [])
    if not cfg.get("enabled", True):
        return {"used": False, "reason": "type_gate_disabled", "type": rec_type, "num_candidates": len(candidates), "config": cfg}
    if RETRIEVAL_ALLOWED_TYPES is not None and rec_type not in set(RETRIEVAL_ALLOWED_TYPES):
        return {
            "used": False,
            "reason": "type_not_allowed",
            "type": rec_type,
            "allowed_types": sorted(RETRIEVAL_ALLOWED_TYPES),
            "num_candidates": len(candidates),
            "config": cfg,
        }
    if len(candidates) < RETRIEVAL_MIN_SOURCE_CANDIDATES:
        return {"used": False, "reason": "source_unseen", "num_candidates": len(candidates), "config": cfg}
    typed = [c for c in candidates if c["type"] == rec_type]
    if cfg.get("require_typed_pool") and not typed:
        return {"used": False, "reason": "typed_pool_missing", "num_candidates": len(candidates), "config": cfg}
    pool = typed or candidates
    if not pool:
        return {"used": False, "reason": "empty_pool", "num_candidates": len(candidates), "config": cfg}

    rec_tokens = query_tokens(rec.get("query_vi"))
    strategy = cfg.get("strategy", RETRIEVAL_STRATEGY)
    if strategy == "source_type_nearest_query":
        selected, ranked = _selected_from_nearest(pool, rec_tokens)
    else:
        selected, ranked = _selected_from_majority(pool, rec_tokens)

    top_count = int(selected["count"])
    other_counts = [int(g["count"]) for g in ranked if not _num_equal(g["answer_num"], selected["answer_num"])]
    second_count = max(other_counts) if other_counts else 0
    majority_frac = top_count / len(pool)
    margin = (top_count - second_count) / len(pool)
    nearest_same = float(selected.get("nearest_jaccard_same_answer") or 0.0)
    model_num = _model_num(model_item)
    model_agrees = _num_equal(model_num, selected["answer_num"])
    confidence_passed = (
        majority_frac >= float(cfg.get("min_majority_frac", 0.0))
        and margin >= float(cfg.get("min_margin", 0.0))
        and nearest_same >= float(cfg.get("min_nearest_jaccard", 0.0))
    )
    agreement_bypass = (
        bool(cfg.get("use_model_agreement_gate", False))
        and bool(cfg.get("model_agreement_bypass_confidence", False))
        and model_agrees
    )
    if not (confidence_passed or agreement_bypass):
        return {
            "used": False,
            "reason": "low_confidence_model_disagree" if model_num is not None and not model_agrees else "low_confidence",
            "strategy": strategy,
            "type": rec_type,
            "num_candidates": len(candidates),
            "pool_candidates": len(pool),
            "typed_pool": bool(typed),
            "majority_count": top_count,
            "second_count": second_count,
            "majority_frac": majority_frac,
            "top1_top2_margin": margin,
            "nearest_jaccard_same_answer": nearest_same,
            "model_num": model_num,
            "model_agrees": model_agrees,
            "confidence_passed": confidence_passed,
            "config": cfg,
        }

    return {
        "used": True,
        "strategy": strategy,
        "canonical_answer": selected["canonical_answer"],
        "answer_num": selected["answer_num"],
        "num_candidates": len(candidates),
        "pool_candidates": len(pool),
        "typed_pool": bool(typed),
        "majority_count": top_count,
        "second_count": second_count,
        "majority_frac": majority_frac,
        "top1_top2_margin": margin,
        "nearest_jaccard_same_answer": nearest_same,
        "nearest_query_jaccard": selected.get("nearest_query_jaccard"),
        "model_num": model_num,
        "model_agrees": model_agrees,
        "confidence_passed": confidence_passed,
        "agreement_bypass": agreement_bypass,
        "config": cfg,
    }

def build_hybrid_outputs(records: list[dict], model_outputs: list[dict], retrieval_index: dict, gate_config_by_type: dict | None = None, *, debug_sample: int = RETRIEVAL_DEBUG_SAMPLE):
    outputs = []
    decisions = []
    used = 0
    changed = 0
    by_type = defaultdict(lambda: {"n": 0, "retrieval_used": 0, "model_fallback": 0})
    reason_counts = Counter()
    agreement_counts = Counter()
    for idx, (rec, model_item) in enumerate(zip(records, model_outputs)):
        decision = retrieve_answer_for_record(rec, retrieval_index, model_item=model_item, gate_config_by_type=gate_config_by_type)
        rec_type = rec.get("type") or "unknown"
        by_type[rec_type]["n"] += 1
        before = model_item.get("model_output", "")
        item = {
            "id": model_item.get("id", rec.get("id", idx)),
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "model_output": before,
        }
        if decision.get("used"):
            item["model_output"] = build_answer_only_target(decision["canonical_answer"])
            used += 1
            changed += int(item["model_output"] != before)
            by_type[rec_type]["retrieval_used"] += 1
            agreement_counts["model_agrees" if decision.get("model_agrees") else "model_disagrees_or_unparseable"] += 1
        else:
            by_type[rec_type]["model_fallback"] += 1
            reason_counts[decision.get("reason", "not_used")] += 1
        if idx < debug_sample:
            decisions.append({
                "idx": idx,
                "type": rec.get("type"),
                "query_vi": rec.get("query_vi", "")[:300],
                "model_output_before": before,
                "model_output_after": item["model_output"],
                "decision": decision,
            })
        outputs.append(item)
    summary = {
        "enabled": True,
        "strategy": RETRIEVAL_STRATEGY,
        "allowed_types": None if RETRIEVAL_ALLOWED_TYPES is None else sorted(RETRIEVAL_ALLOWED_TYPES),
        "active_type_gate_config": gate_config_by_type or ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG,
        "n": len(outputs),
        "retrieval_used": used,
        "retrieval_used_pct": used / len(outputs) if outputs else 0.0,
        "output_changed_count": changed,
        "retrieval_used_by_type": dict(sorted((k, dict(v)) for k, v in by_type.items())),
        "fallback_reason_counts": dict(reason_counts.most_common()),
        "agreement_counts": dict(agreement_counts.most_common()),
        "debug_first": decisions,
    }
    return outputs, summary

def apply_hybrid_retrieval(records: list[dict], model_outputs: list[dict], output_path: Path, retrieval_index: dict, decision_report_path: Path | None = None, gate_config_by_type: dict | None = None):
    if not USE_HYBRID_RETRIEVAL:
        output_path.write_text(json.dumps(model_outputs, ensure_ascii=False, indent=2), encoding="utf-8")
        return model_outputs, {"enabled": False, "n": len(model_outputs), "retrieval_used": 0}
    outputs, summary = build_hybrid_outputs(records, model_outputs, retrieval_index, gate_config_by_type)
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    if decision_report_path is not None:
        decision_report_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"[hybrid] wrote {len(outputs)} rows -> {output_path}; retrieval_used={summary['retrieval_used']}; changed={summary['output_changed_count']}")
    return outputs, summary

def _grid_values(grid: dict, name: str, fallback):
    vals = grid.get(name, fallback)
    return list(vals) if isinstance(vals, (list, tuple)) else [vals]

def iter_retrieval_gate_sweep_configs(rec_type: str):
    grid = dict(RETRIEVAL_GATE_SWEEP_GRID.get("_default", {}))
    grid.update(RETRIEVAL_GATE_SWEEP_GRID.get(rec_type, {}))
    strategies = _grid_values(grid, "strategy", ["source_type_majority"])
    majorities = _grid_values(grid, "min_majority_frac", [RETRIEVAL_MIN_MAJORITY_FRAC])
    margins = _grid_values(grid, "min_margin", [RETRIEVAL_MIN_MAJORITY_MARGIN])
    jaccards = _grid_values(grid, "min_nearest_jaccard", [RETRIEVAL_MIN_NEAREST_JACCARD])
    for strategy in strategies:
        for maj in majorities:
            for margin in margins:
                for jac in jaccards:
                    yield {
                        "enabled": True,
                        "strategy": strategy,
                        "min_majority_frac": float(maj),
                        "min_margin": float(margin),
                        "min_nearest_jaccard": float(jac),
                        "require_typed_pool": bool(grid.get("require_typed_pool", False)),
                        "use_model_agreement_gate": RETRIEVAL_USE_MODEL_AGREEMENT_GATE,
                        "model_agreement_bypass_confidence": RETRIEVAL_MODEL_AGREEMENT_BYPASS_CONFIDENCE,
                    }

def run_retrieval_gate_sweep_for_checkpoint(records: list[dict], model_outputs: list[dict], retrieval_index: dict, *, checkpoint_label: str, order: int) -> dict:
    model_report = evaluate_predictions(model_outputs, records)
    selected_by_type = {}
    per_type = {}
    for rec_type in RETRIEVAL_GATE_SWEEP_TYPES:
        model_type_summary = model_report["by_type"].get(rec_type)
        if not model_type_summary:
            continue
        best = None
        candidates = []
        for cfg_idx, cfg in enumerate(iter_retrieval_gate_sweep_configs(rec_type)):
            outputs, summary = build_hybrid_outputs(records, model_outputs, retrieval_index, {rec_type: cfg}, debug_sample=0)
            report = evaluate_predictions(outputs, records)
            type_summary = report["by_type"].get(rec_type, {"raw_score": 0, "bucket_10": 0, "bucket_0": 0, "n": 0})
            used_by_type = summary["retrieval_used_by_type"].get(rec_type, {})
            entry = {
                "config_index": cfg_idx,
                "config": cfg,
                "summary": type_summary,
                "raw_delta_vs_model": type_summary.get("raw_score", 0) - model_type_summary.get("raw_score", 0),
                "exact10_delta_vs_model": type_summary.get("bucket_10", 0) - model_type_summary.get("bucket_10", 0),
                "retrieval_used": used_by_type.get("retrieval_used", 0),
                "fallback": used_by_type.get("model_fallback", 0),
            }
            key = (
                entry["summary"].get("raw_score", -1),
                entry["summary"].get("bucket_10", -1),
                -entry["summary"].get("bucket_0", 10**9),
                -entry["retrieval_used"],
            )
            entry["selection_key"] = list(key)
            candidates.append(entry)
            if best is None or key > tuple(best["selection_key"]):
                best = entry
        if best is None:
            continue
        if best["summary"].get("raw_score", 0) <= model_type_summary.get("raw_score", 0):
            selected_cfg = {"enabled": False, "disabled_reason": "best_sweep_not_better_than_model_only"}
            selected = {
                "selected": {
                    "config": selected_cfg,
                    "summary": model_type_summary,
                    "raw_delta_vs_model": 0,
                    "exact10_delta_vs_model": 0,
                    "selection_note": "disabled because no retrieval gate was better than model-only for this type",
                },
                "model_only": model_type_summary,
                "top_candidates": sorted(candidates, key=lambda x: tuple(x["selection_key"]), reverse=True)[:10],
            }
        else:
            selected_cfg = best["config"]
            selected = {
                "selected": best,
                "model_only": model_type_summary,
                "top_candidates": sorted(candidates, key=lambda x: tuple(x["selection_key"]), reverse=True)[:10],
            }
        selected_by_type[rec_type] = selected_cfg
        per_type[rec_type] = selected

    combined_outputs, combined_decisions = build_hybrid_outputs(records, model_outputs, retrieval_index, selected_by_type)
    combined_report = evaluate_predictions(combined_outputs, records)
    return {
        "checkpoint_label": checkpoint_label,
        "order": order,
        "model_only_summary": model_report["summary"],
        "selected_gate_config_by_type": selected_by_type,
        "per_type": per_type,
        "combined_summary": combined_report["summary"],
        "combined_by_type": combined_report["by_type"],
        "combined_decision_summary": combined_decisions,
    }

def generate_outputs(adapter_dir: Path, records: list[dict], output_path: Path, *, max_new_tokens: int = MAX_NEW_TOKENS, num_beams: int = NUM_BEAMS, retrieval_index: dict | None = None, model_output_path: Path | None = None, decision_report_path: Path | None = None, gate_config_by_type: dict | None = None):
    model_path = model_output_path or output_path
    model_outputs = generate_model_outputs(adapter_dir, records, model_path, max_new_tokens=max_new_tokens, num_beams=num_beams)
    if USE_HYBRID_RETRIEVAL:
        if retrieval_index is None:
            raise ValueError("USE_HYBRID_RETRIEVAL=True but retrieval_index is None")
        outputs, summary = apply_hybrid_retrieval(records, model_outputs, output_path, retrieval_index, decision_report_path, gate_config_by_type)
        return outputs
    if model_path != output_path:
        shutil.copyfile(model_path, output_path)
    return model_outputs

def _adapter_sort_key(path: Path):
    meta_path = path / "checkpoint_meta.json"
    epoch = 10**9
    final = False
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            epoch = float(meta.get("epoch") or 10**9)
            final = bool(meta.get("final"))
        except Exception:
            pass
    return (epoch, int(final), path.name)

def list_stage_checkpoints() -> list[Path]:
    candidates = []
    if CHECKPOINT_ROOT_DIR.exists():
        candidates.extend([p for p in CHECKPOINT_ROOT_DIR.iterdir() if p.is_dir() and has_peft_adapter(p)])
    for p in [STAGE_A_OUTPUT_DIR, SFT_OUTPUT_DIR, FINAL_OUTPUT_DIR]:
        if has_peft_adapter(p):
            candidates.append(p)
    dedup = []
    seen = set()
    for p in candidates:
        key = str(p.resolve())
        if key not in seen:
            dedup.append(p)
            seen.add(key)
    return sorted(dedup, key=_adapter_sort_key)

def _score_tuple(summary: dict, order: int):
    buckets = summary.get("buckets", {})
    exact10 = buckets.get("10", buckets.get(10, 0))
    raw = summary.get("raw_score", -1)
    extractable = summary.get("extractable", -1)
    order_term = -order if CHECKPOINT_TIE_BREAK == "earlier_epoch" else order
    return (raw, exact10, extractable, order_term)

def select_best_checkpoint_on_valid(records_for_selection: list[dict]):
    global CHECKPOINT_SELECTION, ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG
    ckpts = list_stage_checkpoints()
    if not ckpts:
        print("[select] no adapter checkpoints found; keeping current FINAL_OUTPUT_DIR")
        CHECKPOINT_SELECTION = {
            "enabled": True,
            "selection_split": "valid_json",
            "status": "no_checkpoints_found",
            "final_output_dir": str(FINAL_OUTPUT_DIR),
        }
        SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(CHECKPOINT_SELECTION, ensure_ascii=False, indent=2), encoding="utf-8")
        return CHECKPOINT_SELECTION

    CHECKPOINT_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"[select] evaluating {len(ckpts)} checkpoints on {len(records_for_selection)} valid.json rows")

    entries = []
    sweep_entries = []
    best_entry = None
    best_key = None
    for order, ckpt in enumerate(ckpts):
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", ckpt.name)
        out_path = CHECKPOINT_EVAL_DIR / f"valid_output_{order:02d}_{safe_name}.json"
        model_out_path = CHECKPOINT_EVAL_DIR / f"model_valid_output_{order:02d}_{safe_name}.json"
        report_path = CHECKPOINT_EVAL_DIR / f"valid_report_{order:02d}_{safe_name}.json"
        decision_path = CHECKPOINT_EVAL_DIR / f"hybrid_decisions_{order:02d}_{safe_name}.json"
        gate_sweep_path = CHECKPOINT_EVAL_DIR / f"retrieval_gate_sweep_{order:02d}_{safe_name}.json"

        model_outputs = generate_model_outputs(
            ckpt,
            records_for_selection,
            model_out_path,
            max_new_tokens=CHECKPOINT_EVAL_MAX_NEW_TOKENS,
            num_beams=CHECKPOINT_EVAL_NUM_BEAMS,
        )

        gate_config_by_type = None
        gate_sweep = None
        if USE_HYBRID_RETRIEVAL and RETRIEVAL_GATE_SWEEP_ENABLED:
            gate_sweep = run_retrieval_gate_sweep_for_checkpoint(
                records_for_selection,
                model_outputs,
                RETRIEVAL_INDEX_FULL_TRAIN,
                checkpoint_label=ckpt.name,
                order=order,
            )
            gate_config_by_type = gate_sweep["selected_gate_config_by_type"]
            gate_sweep_path.write_text(json.dumps(gate_sweep, ensure_ascii=False, indent=2), encoding="utf-8")
            sweep_entries.append({
                "order": order,
                "label": ckpt.name,
                "path": str(gate_sweep_path),
                "model_only_summary": gate_sweep["model_only_summary"],
                "combined_summary": gate_sweep["combined_summary"],
                "selected_gate_config_by_type": gate_config_by_type,
            })

        if USE_HYBRID_RETRIEVAL:
            outputs, decision_summary = apply_hybrid_retrieval(
                records_for_selection,
                model_outputs,
                out_path,
                RETRIEVAL_INDEX_FULL_TRAIN,
                decision_report_path=decision_path,
                gate_config_by_type=gate_config_by_type,
            )
        else:
            shutil.copyfile(model_out_path, out_path)
            decision_summary = {"enabled": False}

        rep = save_eval_report(out_path, records_for_selection, report_path)
        summary = rep["summary"]
        meta = {}
        meta_path = ckpt / "checkpoint_meta.json"
        if meta_path.exists():
            try:
                meta = json.loads(meta_path.read_text(encoding="utf-8"))
            except Exception as exc:
                meta = {"meta_error": repr(exc)}
        entry = {
            "order": order,
            "label": ckpt.name,
            "adapter_dir": str(ckpt),
            "output_path": str(out_path),
            "model_output_path": str(model_out_path),
            "report_path": str(report_path),
            "hybrid_decision_path": str(decision_path),
            "gate_sweep_path": str(gate_sweep_path) if gate_sweep is not None else None,
            "retrieval_gate_config_by_type": gate_config_by_type,
            "decision_summary": decision_summary,
            "summary": summary,
            "meta": meta,
        }
        key = _score_tuple(summary, order)
        entry["selection_key"] = list(key)
        entries.append(entry)
        print(f"[select] {ckpt.name}: raw={summary['raw_score']} exact10={summary['buckets'].get('10')} extractable={summary['extractable']} key={key}")
        if best_key is None or key > best_key:
            best_key = key
            best_entry = entry
        if not KEEP_CHECKPOINT_EVAL_OUTPUTS:
            for p in [out_path, model_out_path, decision_path, gate_sweep_path]:
                try:
                    p.unlink()
                except Exception:
                    pass

    if best_entry is None:
        raise RuntimeError("Checkpoint selection failed: no best checkpoint")

    ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG = best_entry.get("retrieval_gate_config_by_type") or RETRIEVAL_TYPE_GATE_CONFIG
    SELECTED_RETRIEVAL_GATE_CONFIG_PATH.write_text(json.dumps(ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG, ensure_ascii=False, indent=2), encoding="utf-8")
    RETRIEVAL_GATE_SWEEP_REPORT_PATH.write_text(json.dumps({
        "enabled": RETRIEVAL_GATE_SWEEP_ENABLED,
        "sweep_types": RETRIEVAL_GATE_SWEEP_TYPES,
        "selected_checkpoint_label": best_entry["label"],
        "selected_gate_config_by_type": ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG,
        "entries": sweep_entries,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    selected_dir = Path(best_entry["adapter_dir"])
    if FINAL_OUTPUT_DIR.exists():
        shutil.rmtree(FINAL_OUTPUT_DIR)
    shutil.copytree(selected_dir, FINAL_OUTPUT_DIR)
    final_hash = sha256_dir(FINAL_OUTPUT_DIR)
    (FINAL_OUTPUT_DIR / "model_hash.txt").write_text(final_hash + "\n", encoding="utf-8")

    CHECKPOINT_SELECTION = {
        "enabled": True,
        "selection_split": "valid_json",
        "selection_metric": "max(valid_json_hybrid_raw_score_after_type_gate_sweep, exact10, extractable, tie_break)" if RETRIEVAL_GATE_SWEEP_ENABLED else "max(valid_json_gated_hybrid_raw_score, exact10, extractable, tie_break)",
        "tie_break": CHECKPOINT_TIE_BREAK,
        "hybrid_retrieval": USE_HYBRID_RETRIEVAL,
        "retrieval_allowed_types": RETRIEVAL_ALLOWED_TYPES,
        "retrieval_gate_sweep_enabled": RETRIEVAL_GATE_SWEEP_ENABLED,
        "selected_retrieval_gate_config_by_type": ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG,
        "status": "selected",
        "eval_n": len(records_for_selection),
        "num_checkpoints": len(entries),
        "selected": best_entry,
        "selected_adapter_dir": str(selected_dir),
        "final_output_dir": str(FINAL_OUTPUT_DIR),
        "final_sha256": final_hash,
        "all_checkpoints": entries,
    }
    CHECKPOINT_SELECTION_REPORT_PATH.write_text(json.dumps(CHECKPOINT_SELECTION, ensure_ascii=False, indent=2), encoding="utf-8")
    SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(CHECKPOINT_SELECTION["selected"], ensure_ascii=False, indent=2), encoding="utf-8")
    print("[select] selected:", best_entry["label"], best_entry["summary"])
    print("[select] active retrieval gate:", ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG)
    print("[select] copied to:", FINAL_OUTPUT_DIR, "sha256=", final_hash)
    return CHECKPOINT_SELECTION

def selected_epoch_count(selection: dict) -> float:
    selected = selection.get("selected") or {}
    meta = selected.get("meta") or {}
    epoch = meta.get("epoch")
    if epoch is not None:
        return float(epoch)
    label = str(selected.get("label") or "")
    m = re.search(r"epoch_(\d+)", label)
    if m:
        return float(m.group(1))
    return float(STAGE_A_EPOCHS)

def retrain_final_on_full_train(selection: dict):
    if not FINAL_RETRAIN_FULL_TRAIN:
        return selection
    if selection.get("status") != "selected":
        print("[final-retrain] skipped because checkpoint selection did not select an epoch")
        return selection

    epochs = selected_epoch_count(selection)
    print("\n" + "=" * 90)
    print(f"[final-retrain] fresh LoRA on full cleaned train: n={len(full_train_stage_a)} epochs={epochs} lr={STAGE_A_LR}")
    if FINAL_OUTPUT_DIR.exists():
        shutil.rmtree(FINAL_OUTPUT_DIR)
    if FULL_TRAIN_OUTPUT_DIR.exists():
        shutil.rmtree(FULL_TRAIN_OUTPUT_DIR)

    final_model = build_lora_model()
    final_model, final_dt = train_lora_stage(
        final_model,
        full_train_stage_a,
        FULL_TRAIN_OUTPUT_DIR,
        MAX_LENGTH_STAGE_A,
        epochs,
        STAGE_A_LR,
        stage_name=FINAL_RETRAIN_STAGE_NAME,
        save_epoch_checkpoints=False,
    )
    shutil.copytree(FULL_TRAIN_OUTPUT_DIR, FINAL_OUTPUT_DIR)
    final_hash = sha256_dir(FINAL_OUTPUT_DIR)
    info = {
        "enabled": True,
        "strategy": "fresh_lora_full_clean_train_after_overlap_valid_epoch_selection",
        "selected_checkpoint_label": selection.get("selected", {}).get("label"),
        "selected_epoch_count": epochs,
        "train_records": len(full_train_stage_a),
        "lr": STAGE_A_LR,
        "max_length": MAX_LENGTH_STAGE_A,
        "output_dir": str(FINAL_OUTPUT_DIR),
        "full_train_output_dir": str(FULL_TRAIN_OUTPUT_DIR),
        "wall_minutes": final_dt / 60,
        "sha256": final_hash,
    }
    FINAL_RETRAIN_INFO_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    selection["final_retrain"] = info
    selection["final_output_dir"] = str(FINAL_OUTPUT_DIR)
    selection["final_sha256"] = final_hash
    CHECKPOINT_SELECTION_REPORT_PATH.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding="utf-8")
    SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(selection.get("selected", {}), ensure_ascii=False, indent=2), encoding="utf-8")
    del final_model
    torch.cuda.empty_cache()
    print("[final-retrain] saved:", FINAL_OUTPUT_DIR, "sha256=", final_hash)
    return selection

CHECKPOINT_SELECTION = {"enabled": False, "status": "not_run"}

if RUN_MODE == "phase1":
    if not VALID_FILE.exists():
        raise FileNotFoundError(f"RUN_MODE='phase1' but missing {VALID_FILE}")
    if SELECT_CHECKPOINTS_ON_OVERLAP_VALID:
        CHECKPOINT_SELECTION = select_best_checkpoint_on_valid(valid_records)

    if (
        CHECKPOINT_SELECTION.get("status") == "selected"
        and CHECKPOINT_SELECTION.get("eval_n") == len(valid_records)
        and CHECKPOINT_EVAL_MAX_NEW_TOKENS == MAX_NEW_TOKENS
        and CHECKPOINT_EVAL_NUM_BEAMS == NUM_BEAMS
    ):
        selected = CHECKPOINT_SELECTION["selected"]
        shutil.copyfile(selected["output_path"], VALID_OUTPUT_PATH)
        if Path(selected["model_output_path"]).exists():
            shutil.copyfile(selected["model_output_path"], MODEL_VALID_OUTPUT_PATH)
        shutil.copyfile(selected["report_path"], VALID_REPORT_PATH)
        valid_rep = json.loads(VALID_REPORT_PATH.read_text(encoding="utf-8"))
        print("[valid_json:selected] reused checkpoint-eval output/report")
    else:
        _ = generate_outputs(
            FINAL_OUTPUT_DIR,
            valid_records,
            VALID_OUTPUT_PATH,
            max_new_tokens=MAX_NEW_TOKENS,
            num_beams=NUM_BEAMS,
            retrieval_index=RETRIEVAL_INDEX_FULL_TRAIN,
            model_output_path=MODEL_VALID_OUTPUT_PATH,
            decision_report_path=HYBRID_DECISION_REPORT_PATH,
        )
        valid_rep = save_eval_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)

    if USE_HYBRID_RETRIEVAL and MODEL_VALID_OUTPUT_PATH.exists():
        model_valid_rep = save_eval_report(MODEL_VALID_OUTPUT_PATH, valid_records, MODEL_VALID_REPORT_PATH)
        print("[valid_json:model_only]", model_valid_rep["summary"])
    print("[valid_json:selected]", valid_rep["summary"])
    print("Reference valid.json Score /10:", valid_rep["summary"]["score_10"])

elif RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"RUN_MODE='phase2' but missing {TEST_FILE}")
    if not VALID_FILE.exists():
        raise FileNotFoundError(f"RUN_MODE='phase2' requires {VALID_FILE} for checkpoint selection")
    if SELECT_CHECKPOINTS_ON_OVERLAP_VALID:
        CHECKPOINT_SELECTION = select_best_checkpoint_on_valid(valid_records)
    test_records = load_records(TEST_FILE)
    _ = generate_outputs(
        FINAL_OUTPUT_DIR,
        test_records,
        TEST_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        retrieval_index=RETRIEVAL_INDEX_FULL_TRAIN,
        model_output_path=MODEL_TEST_OUTPUT_PATH,
        decision_report_path=HYBRID_DECISION_REPORT_PATH,
    )
    print("[phase2] wrote", TEST_OUTPUT_PATH)
else:
    raise ValueError(f"Unknown RUN_MODE={RUN_MODE}")



[retrieval] overlap-train source groups: 12410
[retrieval] full-train source groups: 12410
[select] evaluating 11 checkpoints on 1000 valid.json rows


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[infer] base + adapter: /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_checkpoints/epoch_01
[infer] merged LoRA adapter


...

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[infer] base + adapter: /kaggle/working/gpt2_math_lora_v145_gate_sweep_valid_select_sft
[infer] merged LoRA adapter


generate:gpt2_math_lora_v145_gate_sweep_valid_select_sft:   0%|          | 0/1000 [00:00<?, ?it/s]

[infer] wrote 1000 model rows -> /kaggle/working/v145_gate_sweep_valid_select_checkpoint_eval_valid_json/model_valid_output_10_gpt2_math_lora_v145_gate_sweep_valid_select_sft.json
[hybrid] wrote 1000 rows -> /kaggle/working/v145_gate_sweep_valid_select_checkpoint_eval_valid_json/valid_output_10_gpt2_math_lora_v145_gate_sweep_valid_select_sft.json; retrieval_used=622; changed=237
[select] gpt2_math_lora_v145_gate_sweep_valid_select_sft: raw=6807 exact10=None extractable=996 key=(6807, 664, 996, -10)
[select] selected: epoch_08 {'n': 1000, 'raw_score': 6807, 'max_raw_score': 10000, 'score_10': 6.807, 'score_pct': 0.6807, 'extractable': 996, 'numeric_pairs': 973, 'buckets': {10: 664, 5: 13, 1: 102, 0: 221}, 'rel_error_mean': 2.9660471544394866}
[select] active retrieval gate: {'GSM_Rephrased': {'enabled': True, 'strategy': 'source_type_majority', 'min_majority_frac': 0.34, 'min_margin': 0.33, 'min_nearest_jaccard': 0.0, 'require_typed_pool': False, 'use_model_agreement_gate': True, 'model

In [9]:
# ============================================================
# 8. Output manifest
# ============================================================
def _path_exists_str(p):
    try:
        return Path(p).exists()
    except Exception:
        return False

def _maybe_json_summary(path: Path):
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8")).get("summary")
    except Exception as exc:
        return {"error": repr(exc)}

manifest = {
    "notebook_version": "v14_5_gate_sweep_full_train_valid_select",
    "run_mode": RUN_MODE,
    "created_at_unix": time.time(),
    "use_kd": USE_KD,
    "data": {
        "train_file": str(TRAIN_FILE),
        "valid_file": str(VALID_FILE),
        "test_file": str(TEST_FILE),
        "valid_overlap_audit": str(VALID_OVERLAP_AUDIT_PATH),
        "query_disjoint_split_report": str(QUERY_DISJOINT_SPLIT_PATH),
        "overlap_valid_eval_n": len(globals().get("overlap_valid_eval_records", [])),
    },
    "config": {
        "prompt_template": PROMPT_TEMPLATE,
        "safe_eos_id": SAFE_EOS_ID,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_valid_samples": MAX_VALID_SAMPLES,
        "drop_exact_duplicates": DROP_EXACT_DUPLICATES,
        "drop_non_extractable": DROP_NON_EXTRACTABLE,
        "overlap_valid_query_fraction": OVERLAP_VALID_QUERY_FRACTION,
        "overlap_valid_max_eval_records": OVERLAP_VALID_MAX_EVAL_RECORDS,
        "source_group_key_fields": SOURCE_GROUP_KEY_FIELDS,
        "stage_a_epochs": STAGE_A_EPOCHS,
        "stage_a_lr": STAGE_A_LR,
        "max_length_stage_a": MAX_LENGTH_STAGE_A,
        "per_device_batch_size": PER_DEVICE_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "warmup_ratio": WARMUP_RATIO,
        "weight_decay": WEIGHT_DECAY,
        "seed": SEED,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "lora_target_modules": LORA_TARGET_MODULES,
        "max_new_tokens": MAX_NEW_TOKENS,
        "num_beams": NUM_BEAMS,
        "decode_batch_size": DECODE_BATCH_SIZE,
        "no_repeat_ngram": NO_REPEAT_NGRAM,
        "repetition_penalty": REPETITION_PENALTY,
        "length_penalty": LENGTH_PENALTY,
        "sanitize_to_answer_only": SANITIZE_TO_ANSWER_ONLY,
        "infer_fp16": INFER_FP16,
        "select_checkpoints_on_overlap_valid": SELECT_CHECKPOINTS_ON_OVERLAP_VALID,
        "checkpoint_selection_split_name": CHECKPOINT_SELECTION_SPLIT_NAME,
        "checkpoint_eval_num_beams": CHECKPOINT_EVAL_NUM_BEAMS,
        "checkpoint_eval_max_new_tokens": CHECKPOINT_EVAL_MAX_NEW_TOKENS,
        "checkpoint_tie_break": CHECKPOINT_TIE_BREAK,
        "use_hybrid_retrieval": USE_HYBRID_RETRIEVAL,
        "retrieval_strategy": RETRIEVAL_STRATEGY,
        "retrieval_min_majority_frac": RETRIEVAL_MIN_MAJORITY_FRAC,
        "retrieval_min_majority_margin": RETRIEVAL_MIN_MAJORITY_MARGIN,
        "retrieval_min_nearest_jaccard": RETRIEVAL_MIN_NEAREST_JACCARD,
        "retrieval_allowed_types": RETRIEVAL_ALLOWED_TYPES,
        "retrieval_type_gate_config": RETRIEVAL_TYPE_GATE_CONFIG,
        "active_retrieval_type_gate_config": ACTIVE_RETRIEVAL_TYPE_GATE_CONFIG,
        "retrieval_gate_sweep_enabled": RETRIEVAL_GATE_SWEEP_ENABLED,
        "retrieval_gate_sweep_types": RETRIEVAL_GATE_SWEEP_TYPES,
        "retrieval_use_full_train_for_reference_valid": RETRIEVAL_USE_FULL_TRAIN_FOR_REFERENCE_VALID,
        "final_retrain_full_train": FINAL_RETRAIN_FULL_TRAIN,
        "final_retrain_stage_name": FINAL_RETRAIN_STAGE_NAME,
    },
    "dirs": {
        "stage_a_output_dir": str(STAGE_A_OUTPUT_DIR),
        "sft_output_dir": str(SFT_OUTPUT_DIR),
        "final_output_dir": str(FINAL_OUTPUT_DIR),
        "full_train_output_dir": str(FULL_TRAIN_OUTPUT_DIR),
        "checkpoint_root_dir": str(CHECKPOINT_ROOT_DIR),
        "checkpoint_eval_dir": str(CHECKPOINT_EVAL_DIR),
    },
    "outputs": {
        "overlap_valid_output": str(OVERLAP_VALID_OUTPUT_PATH),
        "overlap_valid_report": str(OVERLAP_VALID_REPORT_PATH),
        "model_overlap_valid_output": str(MODEL_OVERLAP_VALID_OUTPUT_PATH),
        "model_overlap_valid_report": str(MODEL_OVERLAP_VALID_REPORT_PATH),
        "valid_output": str(VALID_OUTPUT_PATH),
        "valid_report": str(VALID_REPORT_PATH),
        "model_valid_output": str(MODEL_VALID_OUTPUT_PATH),
        "model_valid_report": str(MODEL_VALID_REPORT_PATH),
        "hybrid_decision_report": str(HYBRID_DECISION_REPORT_PATH),
        "retrieval_gate_sweep_report": str(RETRIEVAL_GATE_SWEEP_REPORT_PATH),
        "selected_retrieval_gate_config": str(SELECTED_RETRIEVAL_GATE_CONFIG_PATH),
        "final_retrain_info": str(FINAL_RETRAIN_INFO_PATH),
        "checkpoint_selection_report": str(CHECKPOINT_SELECTION_REPORT_PATH),
        "selected_checkpoint_info": str(SELECTED_CHECKPOINT_INFO_PATH),
        "test_predictions": str(TEST_OUTPUT_PATH),
        "model_test_predictions": str(MODEL_TEST_OUTPUT_PATH),
    },
    "query_disjoint_split": globals().get("query_disjoint_split_report"),
    "checkpoint_selection": globals().get("CHECKPOINT_SELECTION", {"enabled": False, "status": "missing_global"}),
    "overlap_valid_summary": _maybe_json_summary(OVERLAP_VALID_REPORT_PATH),
    "reference_valid_summary": _maybe_json_summary(VALID_REPORT_PATH),
    "model_reference_valid_summary": _maybe_json_summary(MODEL_VALID_REPORT_PATH),
    "final_output_dir_exists": _path_exists_str(FINAL_OUTPUT_DIR),
}
manifest_path = WORKING_DIR / "v14_5_gate_sweep_full_train_valid_select_manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print("[manifest] wrote", manifest_path)


{
  "notebook_version": "v14_5_gate_sweep_full_train_valid_select",
  "run_mode": "phase1",
  "created_at_unix": 1780350506.4717324,
  "use_kd": false,
  "data": {
    "train_file": "/kaggle/input/datasets/kimanh2002/dataset-math/train.json",
    "valid_file": "/kaggle/input/datasets/kimanh2002/dataset-math/valid.json",
    "test_file": "/kaggle/input/datasets/kimanh2002/dataset-math/test.json",
    "valid_overlap_audit": "/kaggle/working/valid_overlap_audit.json",
    "query_disjoint_split_report": "/kaggle/working/query_disjoint_split_report.json",
    "overlap_valid_eval_n": 0
  },
  "config": {
    "prompt_template": "Dạng: {type}\nBài toán: {q}\nLời giải: ",
    "safe_eos_id": 50256,
    "max_train_samples": null,
    "max_valid_samples": null,
    "drop_exact_duplicates": true,
    "drop_non_extractable": true,
    "overlap_valid_query_fraction": 0.1,
    "overlap_valid_max_eval_records": 1000,
    "source_group_key_fields": [
      "original_question_en",
      "original_questio